# Sentence Transformer: Setup, Imports

In [13]:
import random, math
import time
import numpy as np
import pandas as pd
import collections
from collections import Counter

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

# Transformers imports
from transformers import BertModel, BertTokenizer, BertConfig

# Set seeds for reproducibility
np.random.seed(100)
torch.manual_seed(100)
random.seed(100)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(100)

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

bert_dropout = 0.1 # From later in the notebook
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
config = BertConfig.from_pretrained('bert-base-uncased',
                                    hidden_dropout_prob=bert_dropout,
                                    attention_probs_dropout_prob=bert_dropout)

bert_base = BertModel.from_pretrained('bert-base-uncased', config=config)
bert_base.to(device)

class CustomVocab:
    def __init__(self, counter, min_freq=1, unknown_token='<unk>'):
        self.unknown_token = unknown_token
        self._token_to_idx = {unknown_token: 0}
        self._idx_to_token = [unknown_token]

        idx = 1
        for token, freq in counter.items():
            if freq >= min_freq:
                if token not in self._token_to_idx:
                    self._token_to_idx[token] = idx
                    self._idx_to_token.append(token)
                    idx += 1
        self._unknown_idx = self._token_to_idx[unknown_token]

    def __getitem__(self, token):
        return self._token_to_idx.get(token, self._unknown_idx)

    def __len__(self):
        return len(self._idx_to_token)

    def __repr__(self):
        return f"CustomVocab(size={len(self)})"

def load_tsv_data(filepath):
    # The original code used field_indices=[1, 2, ..., 13]
    # This implies skipping the first column (index 0) and reading the next 13.
    try:
        data = pd.read_csv(filepath,
                           sep='\t',
                           header=None,
                        #    usecols=range(1, 14),
                           on_bad_lines='skip',
                           encoding='utf-8')
        data.dropna(subset=[2], inplace=True)
        # Convert to list of lists to match gluonnlp.data.TSVDataset output
        return data.values.tolist()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return []
    
def analyze_dataset(raw_data):
    print("analyzing raw data set...\n <><><><><><><><><><> \n")

    label_cnt = Counter()
    topic_cnt = Counter()
    author_cnt = Counter()
    job_cnt = Counter()
    location_cnt = Counter()
    affiliation_cnt = Counter()

    # Try to extract the valid labels, topics, authors,
    for ele in raw_data:
        # Ensure element has enough columns
        if len(ele) < 13:
            continue

        try:
            label, statement, topics, author, job, location, affiliation,cnt_barely, cnt_false, cnt_half, cnt_mostly, cnt_pants_on_fire, venue_context, justification = [str(e) for e in ele[2:16]]

            label_cnt[label] += 1
            for topic in topics.lower().split(','):
                topic_cnt[topic.strip()] += 1
            author_cnt[author.lower()] += 1
            job_cnt[job.lower()] += 1
            location_cnt[location.lower()] += 1
            affiliation_cnt[affiliation.lower()] += 1
        except Exception as e:
            print(f"Error processing row: {ele} | Error: {e}")

    label_map = {ele: i for i, ele in enumerate(label_cnt.keys())}
    print("label map:",label_map)

    #!consider tuning the min_freq param
    # Replace gluonnlp.Vocab with CustomVocab
    topic_vocab = CustomVocab(topic_cnt, min_freq=100)
    author_vocab = CustomVocab(author_cnt, min_freq=50)
    job_vocab = CustomVocab(job_cnt, min_freq=50)
    location_vocab = CustomVocab(location_cnt, min_freq=50)
    affiliation_vocab = CustomVocab(affiliation_cnt, min_freq=50)

    print(topic_vocab)
    print(author_vocab)
    print(job_vocab)
    print(location_vocab)
    print(affiliation_vocab)
    return topic_vocab, author_vocab, job_vocab, location_vocab, affiliation_vocab, label_map

train_dataset_raw = load_tsv_data('../data/liar-plus/train2.tsv')
topic_vocab, author_vocab, job_vocab, location_vocab, affiliation_vocab, label_map = analyze_dataset(train_dataset_raw)


def feature_extraction_transform(data):
    # Transform label into position / negative
    try:
        label, statement, topics, author, job, location, affiliation,cnt_barely, cnt_false, cnt_half, cnt_mostly, cnt_pants_on_fire, venue_context, justification = [str(e) for e in data[2:16]]
    except Exception as e:
        print(f"Error in feature_extraction_transform: {e}, data: {data}")
        return None # Return None to filter out this bad data

    topic_one_encoding = np.zeros(shape=(len(topic_vocab,)), dtype=np.float32)
    topic_ids = [topic_vocab[t.strip()] for t in topics.lower().split(',')]
    topic_one_encoding[topic_ids] = 1
    if len(topic_ids) > 0 and topic_one_encoding.sum() > 0:
        topic_one_encoding /= topic_one_encoding.sum()

    author_id = author_vocab[author.lower()]
    job_id = job_vocab[job.lower()]
    location_id = location_vocab[location.lower()]
    affiliation_id = affiliation_vocab[affiliation.lower()]

    try:
        cnt_barely_f = float(cnt_barely)
        cnt_false_f = float(cnt_false)
        cnt_half_f = float(cnt_half)
        cnt_mostly_f = float(cnt_mostly)
        cnt_pants_on_fire_f = float(cnt_pants_on_fire)
    except ValueError:
        # Handle cases where conversion to float fails
        cnt_barely_f = cnt_false_f = cnt_half_f = cnt_mostly_f = cnt_pants_on_fire_f = 0.0

    cnt_total = cnt_barely_f + cnt_false_f + cnt_half_f + cnt_mostly_f + cnt_pants_on_fire_f

    if cnt_total > 0 :
        proportion = [cnt_barely_f / cnt_total,
                      cnt_false_f / cnt_total,
                      cnt_half_f / cnt_total,
                      cnt_mostly_f / cnt_total,
                      cnt_pants_on_fire_f / cnt_total]
    else:
        proportion = [0.0, 0.0, 0.0, 0.0, 0.0]

    cnt_uncertainty = 1.0 / (cnt_total + 1.0)
    history_of_truth = np.array(proportion + [cnt_uncertainty], dtype=np.float32)
    venue_feature = 0 # venue_feature = f(venue_context), keep it as a vector

    return (statement, topic_one_encoding, author_id, job_id, location_id, affiliation_id,
            history_of_truth, venue_feature, label_map[label])

class BERTClassifier(nn.Module):
    def __init__(self, bert, num_topics, num_authors, num_jobs, num_locations, num_affiliations, num_classes,
                 embed_dim,
                 author_dropout, author_mlp_layers, author_mlp_hidden,
                 history_dropout, history_mlp_layers, history_mlp_hidden):
        super(BERTClassifier, self).__init__()
        self.bert = bert

        # Note: BERT hidden size is 768
        bert_hidden_size = bert.config.hidden_size

        self.topic_embed = nn.Linear(num_topics, embed_dim) # Use one-hot encoding for topics
        self.author_embed = nn.Embedding(num_embeddings=num_authors, embedding_dim=embed_dim)
        self.job_embed = nn.Embedding(num_embeddings=num_jobs, embedding_dim=embed_dim)
        self.location_embed = nn.Embedding(num_embeddings=num_locations, embedding_dim=embed_dim)
        self.affiliation_embed = nn.Embedding(num_embeddings=num_affiliations, embedding_dim=embed_dim)

        author_feature_map_layers = []
        author_feature_map_layers.append(nn.Dropout(author_dropout))
        author_input_dim = embed_dim * 5 # topic + author + job + location + affiliation
        for _ in range(author_mlp_layers):
            author_feature_map_layers.append(nn.Linear(author_input_dim, author_mlp_hidden))
            author_feature_map_layers.append(nn.LeakyReLU(0.1))
            author_feature_map_layers.append(nn.Dropout(author_dropout))
            author_input_dim = author_mlp_hidden # Input for next layer
        self.author_feature_map = nn.Sequential(*author_feature_map_layers)

        history_feature_map_layers = []
        history_input_dim = 6 # 5 proportions + 1 uncertainty
        for _ in range(history_mlp_layers):
            history_feature_map_layers.append(nn.Linear(history_input_dim, history_mlp_hidden))
            history_feature_map_layers.append(nn.LeakyReLU(0.1))
            history_feature_map_layers.append(nn.Dropout(history_dropout))
            history_input_dim = history_mlp_hidden # Input for next layer
        self.history_feature_map = nn.Sequential(*history_feature_map_layers)

        # extra layer used for classification
        classifier_input_dim = bert_hidden_size + author_input_dim + history_input_dim
        self.classifier = nn.Linear(classifier_input_dim, num_classes)


    def forward(self, inputs, segment_types, attention_mask,
                topic_one_hot, author_id, job_id, location_id, affiliation_id, history_feature):

        # Encode the news representation using BERT
        # seq_len is replaced by attention_mask
        outputs = self.bert(input_ids=inputs,
                            token_type_ids=segment_types,
                            attention_mask=attention_mask)

        # We use the pooler_output, which corresponds to the [CLS] token
        cls_encoding = outputs.pooler_output

        #dataset specific features:
        topic_fea = self.topic_embed(topic_one_hot)
        author_fea = self.author_embed(author_id)
        job_fea = self.job_embed(job_id)
        location_fea = self.location_embed(location_id)
        affiliation_fea = self.affiliation_embed(affiliation_id)

        # Concat author-related features
        author_features_combined = torch.cat((topic_fea, author_fea, job_fea, location_fea, affiliation_fea), dim=-1)
        author_feature = self.author_feature_map(author_features_combined)

        history_feature = self.history_feature_map(history_feature)

        # Concat all features for final classification
        combined_features = torch.cat((cls_encoding, author_feature, history_feature), dim=-1)

        return self.classifier(combined_features)
    
def transform_fn(text, topic_one_encoding, author_id, job_id, location_id, affiliation_id, history_feature, venue_ids, label):
    max_len = 256

    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,  # Adds [CLS] and [SEP]
        max_length=max_len,
        truncation=True,
        padding=False,            # We will pad in the collate_fn
        return_token_type_ids=True
    )

    data = np.array(encoding['input_ids'], dtype='int64')
    length = np.array(len(data), dtype='int64') # This is now just the length, not a tensor
    segment_type = np.array(encoding['token_type_ids'], dtype='int64')

    # history_feature is already a numpy array, just ensure type
    history_feature = np.array(history_feature, dtype=np.float32)

    # Return all features as standard types for batching
    return (data, length, segment_type, topic_one_encoding, author_id, job_id,
            location_id, affiliation_id, history_feature, label)

class ListDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

padding_id = tokenizer.pad_token_id
print(f"Padding token ID: {padding_id}")

def collate_fn(batch):
    (data, lengths, segment_type, topic_one_encoding, author_id, job_id,
     location_id, affiliation_id, history_feature, label) = zip(*batch)

    # Pad sequences
    data_padded = pad_sequence(
        [torch.as_tensor(d, dtype=torch.long) for d in data],
        batch_first=True, padding_value=padding_id
    )

    seg_types_padded = pad_sequence(
        [torch.as_tensor(s, dtype=torch.long) for s in segment_type],
        batch_first=True, padding_value=0
    )

    attention_mask = (data_padded != padding_id).long()

    # Convert to tensors (no stacking needed if already numbers)
    lengths = torch.tensor([int(l) for l in lengths], dtype=torch.long)
    topic_one_encoding = torch.as_tensor(topic_one_encoding, dtype=torch.float32)
    author_id = torch.as_tensor(author_id, dtype=torch.long)
    job_id = torch.as_tensor(job_id, dtype=torch.long)
    location_id = torch.as_tensor(location_id, dtype=torch.long)
    affiliation_id = torch.as_tensor(affiliation_id, dtype=torch.long)
    history_feature = torch.as_tensor(history_feature, dtype=torch.float32)
    label = torch.as_tensor(label, dtype=torch.long)

    # Optional pinning (usually safe to skip unless you explicitly use pin_memory=True in DataLoader)
    # for t in [data_padded, seg_types_padded, attention_mask, lengths, topic_one_encoding,
    #           author_id, job_id, location_id, affiliation_id, history_feature, label]:
    #     t = t.pin_memory()

    return (data_padded, seg_types_padded, attention_mask,
            topic_one_encoding, author_id, job_id, location_id,
            affiliation_id, history_feature, label)

def evaluate_loop(net, eval_data, device):
    net.eval()
    total_num = 0
    hits = 0
    running_loss = 0.0

    with torch.no_grad():
        for batch_idx, (inputs, token_types, attention_mask, topic_one_encoding,
                        author_id, job_id, location_id, affiliation_id, history, label) in enumerate(eval_data):

            # Move tensors to device
            inputs = inputs.to(device, non_blocking=True)
            token_types = token_types.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)
            topic_one_encoding = topic_one_encoding.to(device, non_blocking=True)
            author_id = author_id.to(device, non_blocking=True)
            job_id = job_id.to(device, non_blocking=True)
            location_id = location_id.to(device, non_blocking=True)
            affiliation_id = affiliation_id.to(device, non_blocking=True)
            history = history.to(device, non_blocking=True)

            out = net(inputs, token_types, attention_mask, topic_one_encoding,
                      author_id, job_id, location_id, affiliation_id, history)

            total_num += out.shape[0]
            hits += (out.argmax(dim=-1) == label).sum().item()

    return hits / float(total_num)

Using device: cuda
analyzing raw data set...
 <><><><><><><><><><> 

label map: {'false': 0, 'half-true': 1, 'mostly-true': 2, 'true': 3, 'barely-true': 4, 'pants-fire': 5}
CustomVocab(size=60)
CustomVocab(size=22)
CustomVocab(size=22)
CustomVocab(size=28)
CustomVocab(size=7)
Padding token ID: 0


# Sentence Transformer: Test Set Evaluation

In [ ]:
test_dataset_raw = load_tsv_data('../data/liar-plus/test2.tsv')
test_dataset = [d for d in [feature_extraction_transform(data) for data in test_dataset_raw] if d is not None]
test_dataset_bert = [transform_fn(*sample) for sample in test_dataset]
test_data_torch = ListDataset(test_dataset_bert)
batch_size = 16
test_data = DataLoader(test_data_torch,
                       batch_size=batch_size,
                       shuffle=False,
                       collate_fn=collate_fn,
                       num_workers=0,
                       pin_memory=False)

# embed_dim = 16
# author_dropout = 0.1
# author_mlp_layers = 2
# author_mlp_hidden = 128
# history_dropout = 0.1
# history_mlp_layers = 2
# history_mlp_hidden = 128
embed_dim = 16
author_dropout = 0.1
author_mlp_layers = 2
author_mlp_hidden = 128
history_dropout = 0.1
history_mlp_layers = 2
history_mlp_hidden = 128

net_best = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=embed_dim,
    author_dropout=author_dropout,
    author_mlp_layers=author_mlp_layers,
    author_mlp_hidden=author_mlp_hidden,
    history_dropout=history_dropout,
    history_mlp_layers=history_mlp_layers,
    history_mlp_hidden=history_mlp_hidden)

# Load the saved state dictionary
# state_dict = torch.load(
#     '/Users/ryanxavier/Downloads/ai4good/Untitled/notebooks/bert_liar_liar_best.pth',
#     map_location=torch.device('cpu')
# )
state_dict = torch.load(
    '../checkpoints/best.pth'
)
net_best.load_state_dict(state_dict)

net_best.to(device) # Move to device


def evaluate_loop(net, eval_data, device):
    net.eval()
    total_num = 0
    hits = 0
    running_loss = 0.0

    with torch.no_grad():
        for batch_idx, (inputs, token_types, attention_mask, topic_one_encoding,
                        author_id, job_id, location_id, affiliation_id, history, label) in enumerate(eval_data):

            # Move tensors to device
            inputs = inputs.to(device, non_blocking=True)
            token_types = token_types.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            label = label.to(device, non_blocking=True)
            topic_one_encoding = topic_one_encoding.to(device, non_blocking=True)
            author_id = author_id.to(device, non_blocking=True)
            job_id = job_id.to(device, non_blocking=True)
            location_id = location_id.to(device, non_blocking=True)
            affiliation_id = affiliation_id.to(device, non_blocking=True)
            history = history.to(device, non_blocking=True)

            out = net(inputs, token_types, attention_mask, topic_one_encoding,
                      author_id, job_id, location_id, affiliation_id, history)

            total_num += out.shape[0]
            hits += (out.argmax(dim=-1) == label).sum().item()

    return hits / float(total_num)

# Evaluate
test_acc = evaluate_loop(net_best, test_data, device)
print('Test acc =', test_acc) 

RuntimeError: Error(s) in loading state_dict for BERTClassifier:
	Missing key(s) in state_dict: "topic_embed.weight", "topic_embed.bias", "classifier.weight", "classifier.bias". 
	Unexpected key(s) in state_dict: "topic_embed.0.weight", "topic_embed.0.bias", "topic_embed.2.weight", "topic_embed.2.bias", "classifier.0.weight", "classifier.0.bias", "classifier.3.weight", "classifier.3.bias". 

In [4]:
import os
os.environ["OPENAI_API_KEY"] = "0oaQ7PXB0WabrM1ubjWxAWLRiBXNtPQn"

In [5]:
import os
from openai import OpenAI
import json

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://ellm.nrp-nautilus.io/v1"
)

def infer_metadata_with_ai(statement):
    prompt = f"""
Extract the following fields from the statement below.
Fields 1–7 must always be filled with plausible inferred values. 
Never return "unknown" for these fields. 
Fields 8–12 must always be "0". 
Fields 13 and 14 must always be empty strings.

Required JSON keys in this exact order:

1. label
2. statement
3. topics
4. author
5. job
6. location
7. affiliation
8. cnt_barely       = "0"
9. cnt_false        = "0"
10. cnt_half        = "0"
11. cnt_mostly      = "0"
12. cnt_pants_on_fire = "0"
13. venue_context   = ""
14. justification   = ""

Statement:
\"\"\"{statement}\"\"\"

Return ONLY a JSON object with these exact 14 keys.
"""

    completion = client.chat.completions.create(
        model="gemma3",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}]
    )

    return json.loads(completion.choices[0].message.content)


def load_and_fill_dataset(path):
    df = pd.read_csv(path)
    all_rows = []

    for _, row in df.iterrows():
        statement = row["text"]
        
        ai_fields = infer_metadata_with_ai(statement)

        ordered_list = [
            ai_fields["label"],
            ai_fields["statement"],
            ai_fields["topics"],
            ai_fields["author"],
            ai_fields["job"],
            ai_fields["location"],
            ai_fields["affiliation"],
            "0",   # cnt_barely
            "0",   # cnt_false
            "0",   # cnt_half
            "0",   # cnt_mostly
            "0",   # cnt_pants_on_fire
            "",  # venue_context
            ""   # justification
        ]

        all_rows.append(ordered_list)

    return all_rows



In [6]:
def predict_dataset_labels(path):
    test_dataset_raw = pd.read_csv(path).values #.to_list()
    test_dataset = [d for d in [feature_extraction_transform(data) for data in test_dataset_raw] if d is not None]
    test_dataset_bert = [transform_fn(*sample) for sample in test_dataset]
    test_data_torch = ListDataset(test_dataset_bert)
    batch_size = 16
    test_data = DataLoader(test_data_torch,
                           batch_size=batch_size,
                           shuffle=False,
                           collate_fn=collate_fn,
                           num_workers=0,
                           pin_memory=False)

    # Assuming net_best is already created and loaded
    net_best.eval()
    all_preds = []

    inv_label_map = {v: k for k, v in label_map.items()}  # Map indices -> label strings

    with torch.no_grad():
        for batch_idx, (inputs, token_types, attention_mask, topic_one_encoding,
                        author_id, job_id, location_id, affiliation_id, history, label) in enumerate(test_data):

            # Move tensors to device
            inputs = inputs.to(device)
            token_types = token_types.to(device)
            attention_mask = attention_mask.to(device)
            topic_one_encoding = topic_one_encoding.to(device)
            author_id = author_id.to(device)
            job_id = job_id.to(device)
            location_id = location_id.to(device)
            affiliation_id = affiliation_id.to(device)
            history = history.to(device)

            # Forward pass
            out = net_best(inputs, token_types, attention_mask, topic_one_encoding,
                           author_id, job_id, location_id, affiliation_id, history)

            # Get predicted indices
            preds = out.argmax(dim=-1).cpu().tolist()
            # Map to label strings
            all_preds.extend([inv_label_map[i] for i in preds])

    return all_preds

predict_dataset_labels('data/polifact.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/polifact.csv'

# Sentence Transformer: classify a single article

In [12]:
import torch
import json
import os
from openai import OpenAI

# 1. Define the Label Map (Must match your training configuration)
# This maps the model's integer output back to the readable string.
LABEL_MAP = {
    0: 'false',
    1: 'half-true',
    2: 'mostly-true',
    3: 'true',
    4: 'barely-true',
    5: 'pants-fire'
}
# Inverse map for looking up indices
LABEL_TO_IDX = {v: k for k, v in LABEL_MAP.items()}

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://ellm.nrp-nautilus.io/v1"
)

def to_long_tensor(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device, dtype=torch.long)
    return torch.tensor(x, dtype=torch.long).to(device, dtype=torch.long)

def infer_metadata_with_ai(statement):
    # We update the prompt to force the AI to pick one of the 6 valid labels.
    # This ensures the 'label' column is never invalid/filtered out.
    prompt = f"""
Analyze the statement below. 
1. Extract metadata (author, job, location, affiliation). 
2. Classify the statement into exactly one of these labels: 
   ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire'].

Fields 1-7 must be plausible strings (no "unknown").
Fields 8-12 must be "0".
Fields 13-14 must be empty strings.

Required JSON keys:
1. label (Must be one of: false, half-true, mostly-true, true, barely-true, pants-fire)
2. statement
3. topics (comma separated keywords)
4. author
5. job
6. location
7. affiliation
8. cnt_barely = "0"
9. cnt_false = "0"
10. cnt_half = "0"
11. cnt_mostly = "0"
12. cnt_pants_on_fire = "0"
13. venue_context = ""
14. justification = ""

Statement:
\"\"\"{statement}\"\"\"

Return ONLY a JSON object.
"""

    completion = client.chat.completions.create(
        model="gemma3",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}]
    )

    return json.loads(completion.choices[0].message.content)

def classify_single_article(article_text, net, device):
    ai_fields = infer_metadata_with_ai(article_text)
    
    # Process topics - keep commas since the function expects comma-separated
    topics = ai_fields.get("topics", "news")
    if isinstance(topics, list):
        topics = ",".join([t.strip() for t in topics])
    else:
        topics = str(topics)
    
    valid_labels = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
    temp_label = ai_fields.get("label", "false")
    if temp_label not in valid_labels:
        temp_label = "false"

    # The function expects data[2:16], so we need indices 0, 1, and then 2-15 (14 fields)
    article_row = [
        0,                                             # 0. Row index (dummy)
        "12345.json",                                  # 1. ID
        temp_label,                                    # 2. Label
        str(ai_fields.get("statement", article_text)), # 3. Statement
        topics,                                        # 4. Topics (comma-separated)
        str(ai_fields.get("author", "Anonymous")),     # 5. Author
        str(ai_fields.get("job", "Reporter")),         # 6. Job
        str(ai_fields.get("location", "Unknown")),     # 7. Location
        str(ai_fields.get("affiliation", "Unknown")),  # 8. Affiliation
        "0",                                           # 9. cnt_barely
        "0",                                           # 10. cnt_false
        "0",                                           # 11. cnt_half
        "0",                                           # 12. cnt_mostly
        "0",                                           # 13. cnt_pants_on_fire
        "",                                            # 14. venue_context
        ""                                             # 15. justification
    ]

    sample = feature_extraction_transform(article_row)
    if sample is None:
        raise ValueError(f"Feature extraction failed. Input row: {article_row}")

    transformed = transform_fn(*sample)
    
    inputs, token_types, attention_mask, topic_one_encoding, \
    author_id, job_id, location_id, affiliation_id, history, label = transformed

    
    inputs = to_long_tensor(inputs, device)
    if inputs.dim() == 1:
        inputs = inputs.unsqueeze(0)
    elif inputs.dim() == 0:
        inputs = inputs.unsqueeze(0).unsqueeze(0)
    
    token_types = to_long_tensor(token_types, device)
    if token_types.dim() == 1:
        token_types = token_types.unsqueeze(0)
    elif token_types.dim() == 0:
        token_types = token_types.unsqueeze(0).unsqueeze(0)
    
    attention_mask = to_long_tensor(attention_mask, device)
    if attention_mask.dim() == 1:
        attention_mask = attention_mask.unsqueeze(0)
    elif attention_mask.dim() == 0:
        attention_mask = attention_mask.unsqueeze(0).unsqueeze(0)
    
    topic_one_encoding = torch.tensor(topic_one_encoding, dtype=torch.float, device=device)
    if topic_one_encoding.dim() == 1:
        topic_one_encoding = topic_one_encoding.unsqueeze(0)
    
    author_id = to_long_tensor(author_id, device)
    if author_id.dim() == 0:
        author_id = author_id.unsqueeze(0)
    
    job_id = to_long_tensor(job_id, device)
    if job_id.dim() == 0:
        job_id = job_id.unsqueeze(0)
    
    location_id = to_long_tensor(location_id, device)
    if location_id.dim() == 0:
        location_id = location_id.unsqueeze(0)
    
    affiliation_id = to_long_tensor(affiliation_id, device)
    if affiliation_id.dim() == 0:
        affiliation_id = affiliation_id.unsqueeze(0)
    
    history = torch.tensor(history, dtype=torch.float, device=device)
    if history.dim() == 1:
        history = history.unsqueeze(0)


    
    if token_types.shape != inputs.shape:
        token_types = torch.zeros_like(inputs, dtype=torch.long, device=device)

    net.eval()
    with torch.no_grad():
        logits = net(
            inputs, token_types, attention_mask,
            topic_one_encoding, author_id, job_id,
            location_id, affiliation_id, history
        )

    pred_idx = logits.argmax(dim=-1).item()
    final_label = LABEL_MAP.get(pred_idx, "unknown")
    
    return final_label


article_text = "The economy has grown by 50% this year under my administration."
prediction = classify_single_article(article_text, net_best, device)
print(f"Final Model Prediction: {prediction}")

Final Model Prediction: true


# Load new data

In [ ]:
import sys
sys.path.append('../utils/')
from nlp_utils import * 

df_train = open_data('../data/liar-plus/train2.tsv')
df_test = open_data('../data/liar-plus/test2.tsv')
df_val = open_data('../data/liar-plus/val2.tsv')
df_train.head()

df_train["statement"] = df_train["statement"].astype(str)
df_train.dropna(subset=['statement','label'], inplace=True)
df_train["statement"].apply(clean_text)
df_train = df_train[['statement','label']]

df_val["statement"] = df_val["statement"].astype(str)
df_val.dropna(subset=['statement','label'], inplace=True)
df_val["statement"].apply(clean_text)
df_val = df_val[['statement','label']]

df_test["statement"] = df_test["statement"].astype(str)
df_test.dropna(subset=['statement','label'], inplace=True)
df_test["statement"].apply(clean_text)
df_test = df_test[['statement','label']]

# Spam

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
import torch
import pandas as pd

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def get_spam_scores(text_list, batch_size=16):
    scores = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = spam_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512
        )
        with torch.no_grad():
            outputs = spam_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            scores.extend(probs[:, 1].tolist())
    return scores

df_train['spam_score'] = get_spam_scores(df_train['statement'].tolist())
df_test['spam_score'] = get_spam_scores(df_test['statement'].tolist())
df_val['spam_score'] = get_spam_scores(df_val['statement'].tolist())

spam_scaler = StandardScaler()
df_train['spam_score'] = spam_scaler.fit_transform(df_train[['spam_score']])
df_val['spam_score'] = spam_scaler.transform(df_val[['spam_score']])   
df_test['spam_score'] = spam_scaler.transform(df_test[['spam_score']])

# Political Bias

In [ ]:
import spacy
# spacy.cli.download("en_core_web_md")
datum = df_train.iloc[0]
nlp = spacy.load("en_core_web_md")
doc = nlp(datum['statement'])
doc.vector.shape
statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}

def stat_counter(text):
    if not isinstance(text, str):
        return 0
    doc = nlp(text)
    counter = 0
    for ent in doc.ents: 
        if ent.label_ in statistic_types:
            counter += 1
    return counter

from rapidfuzz import fuzz
conservative_bigrams = pd.read_csv('top_conservative_bigrams.csv')['bigram']
liberal_bigrams = pd.read_csv('top_liberal_bigrams.csv')['bigram']
def match_counter(statement, bigram_list, threshold):
    stat = nlp(str(statement))
    word = [word.text.lower() for word in stat]
    bigram_coll = [''.join(word[i:i+2]) for i in range(len(word)-1)]
    matches = 0
    for bigram in bigram_coll:
        for check in bigram_list:
            if fuzz.ratio(bigram, check) >= threshold:
                matches += 1
                break

    return matches


df_train['statistic_count'] = df_train['statement'].apply(stat_counter)
df_train['conservative_bigram_count'] = df_train['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_train['liberal_bigram_count'] = df_train['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

df_val['statistic_count'] = df_val['statement'].apply(stat_counter)
df_val['conservative_bigram_count'] = df_val['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_val['liberal_bigram_count'] = df_val['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

df_test['statistic_count'] = df_test['statement'].apply(stat_counter)
df_test['conservative_bigram_count'] = df_test['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_test['liberal_bigram_count'] = df_test['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

count_features = ["statistic_count", "conservative_bigram_count", "liberal_bigram_count"]
scaler_counts = StandardScaler()

df_train[count_features] = scaler_counts.fit_transform(df_train[count_features])
df_val[count_features]   = scaler_counts.transform(df_val[count_features])
df_test[count_features]  = scaler_counts.transform(df_test[count_features])

# Sensationalism

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def emotional_intensity_vader(text):
    if not isinstance(text, str):
        return 0.0
    if len(text) == 0:
        return 0.0
    vs = analyzer.polarity_scores(text)
    return abs(vs['compound'])

for df in [df_train, df_test, df_val]:
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)

scaler_vader = StandardScaler()
df_train["emotional_intensity"] = scaler_vader.fit_transform(df_train[["emotional_intensity"]])
df_val["emotional_intensity"]   = scaler_vader.transform(df_val[["emotional_intensity"]])
df_test["emotional_intensity"]  = scaler_vader.transform(df_test[["emotional_intensity"]])

In [ ]:
def add_veracity_features(df):
    df = df.copy()

    # ----- Spam feature -----
    df['spam_score'] = get_spam_scores(df['statement'].tolist())
    df['spam_score'] = spam_scaler.transform(df[['spam_score']])

    # ----- Count-based features -----
    df['statistic_count'] = df['statement'].apply(stat_counter)
    df['conservative_bigram_count'] = df['statement'].apply(
        lambda x: match_counter(x, conservative_bigrams, threshold=70)
    )
    df['liberal_bigram_count'] = df['statement'].apply(
        lambda x: match_counter(x, liberal_bigrams, threshold=70)
    )
    count_features = ["statistic_count", "conservative_bigram_count", "liberal_bigram_count"]
    df[count_features] = scaler_counts.transform(df[count_features])  # use transform, not fit_transform!

    # ----- Emotional intensity -----
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)
    df['emotional_intensity'] = scaler_vader.transform(df[['emotional_intensity']])

    return df

In [ ]:
augmented_train = add_veracity_features(df_train)
augmented_val = add_veracity_features(df_val)
augmented_test = add_veracity_features(df_test)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
augmented_train['label'] = le.fit_transform(augmented_train['label'])
augmented_val['label'] = le.transform(augmented_val['label'])
augmented_test['label'] = le.transform(augmented_test['label'])

# Predictive Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

X_train = augmented_train.drop(columns=['statement', 'label'])
y_train = augmented_train['label']
X_val = augmented_val.drop(columns=['statement', 'label'])
y_val = augmented_val['label']
X_test = augmented_test.drop(columns=['statement', 'label'])
y_test = augmented_test['label']

clf = RandomForestClassifier(n_estimators=600, max_depth=200, random_state=21)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

 barely-true       0.19      0.17      0.18       212
       false       0.21      0.21      0.21       249
   half-true       0.22      0.24      0.23       265
 mostly-true       0.19      0.21      0.20       241
  pants-fire       0.09      0.08      0.08        92
        true       0.20      0.19      0.20       208

    accuracy                           0.20      1267
   macro avg       0.18      0.18      0.18      1267
weighted avg       0.20      0.20      0.20      1267



# Unseen Data

In [ ]:
from ingestor_utils import ingest_articles

unseen_df = ingest_articles("Columbus Day")
augment_unseen_df = add_veracity_features(unseen_df)
X_unseen = augment_unseen_df.drop(columns=['statement','source'])
pred_labels = clf.predict(X_unseen)
unseen_df['predicted_label'] = le.inverse_transform(pred_labels)
unseen_df

,statement,source,predicted_label
0,This story is available exclusively to Busines...,Business Insider,false
1,Texas stays in the playoff chase as the Longho...,ESPN,false
2,"Another wave of ""Red Cup Rebellions"" is coming...",USA Today,false
3,ZANESVILLE – A shooting incident from Septembe...,Zanesvilletimesrecorder.com,false
4,"COLUMBUS, Ohio (AP) — The murder trial of an O...",Yahoo Entertainment,mostly-true
...,...,...,...
68,"By Andrew Sanford | News | November 6, 2025 I’...",Pajiba.com,half-true
69,Ohio State is still 7-0 following the bye week...,Land-Grant Holy Land,false
70,"One of the first things to impress onlookers, ...",BBC News,barely-true
71,Ohio State is the model to follow. Not Indiana...,Fox Sports,false
